# Логистическая регрессия для прогноза сделок

Оцениваем модель на одном отложенном тесте. Для итоговой выгрузки обучаем модель на всех объявлениях с доступными метками. В тесте и в итоговом CSV присваиваем `0` заблокированным объявлениям.

## 1. Загрузить данные

Используем `features.csv` из `feature_engineering.ipynb`. Метки и признаки сопоставляем по `listing_id`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [2]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_PATH = Path("submission_logreg_without_cv_features.csv")

In [3]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.isin(features.index).all()

feature_columns = features.columns.difference(["seller_id", "is_deal"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]
X_train = X.loc[y.index]

print(f"Объявлений в матрице: {len(X)}")
print(f"Размеченных объявлений с признаками: {len(y)}")

Объявлений в матрице: 15274
Размеченных объявлений с признаками: 300


## 2. Создать модель

В числовом блоке заменяем бесконечности пропусками, заполняем пропуски медианой и стандартизируем значения. В категориальном блоке заполняем пропуски самым частым значением и применяем one-hot.

Логистическая регрессия использует Elastic Net с `C=0.8` и `l1_ratio=0.5`.

In [4]:
numeric_columns = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = X_train.columns.difference(numeric_columns, sort=False).tolist()


def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(
    C=0.8,
    l1_ratio=0.5,
    solver="saga",
    max_iter=5000,
    random_state=42,
)
)

## 3. Оценить модель на отложенном тесте

Делим `verified.csv` в пропорции 80/20, сохраняя близкую долю сделок в обеих частях. На тесте для заблокированных объявлений ставим ноль, как и при сохранении итогового CSV.

In [5]:
blocked_ids = listings.index[listings["close_type"] == "заблокировано"]

In [6]:
train_y, test_y = train_test_split(y, test_size=0.2, stratify=y, random_state=42)

test_model = clone(model).fit(X.loc[train_y.index], train_y)
test_probability = pd.Series(
    test_model.predict_proba(X.loc[test_y.index])[:, 1],
    index=test_y.index,
)
test_probability.loc[test_y.index.intersection(blocked_ids)] = 0
test_prediction = (test_probability > 0.5).astype(int)

metrics = pd.Series({
    "F1 по сделкам": f1_score(test_y, test_prediction),
    "Точность (precision)": precision_score(test_y, test_prediction),
    "Полнота (recall)": recall_score(test_y, test_prediction),
    "ROC-AUC": roc_auc_score(test_y, test_probability),
    "Доля верных ответов (accuracy)": accuracy_score(test_y, test_prediction),
}, name="Значение")

print(f"Обучение: {len(train_y)} размеченных объявлений с признаками")
print(f"Тест: {len(test_y)} объявлений, из них {test_y.index.isin(blocked_ids).sum()} заблокировано")
metrics.round(4).to_frame()

Обучение: 240 размеченных объявлений с признаками
Тест: 60 объявлений, из них 1 заблокировано
Out[6]: 
                                Значение
F1 по сделкам                     0.7273
Точность (precision)              0.7619
Полнота (recall)                  0.6957
ROC-AUC                           0.8837
Доля верных ответов (accuracy)    0.8000


## 4. Обучить итоговую модель и получить прогнозы

Обучаем модель на всех объявлениях с доступными метками. Для каждой строки выбранного CSV получаем прогноз `is_deal` равный `0` или `1`.

In [7]:
model.fit(X_train, y)
predictions = X.index.to_frame(index=False).assign(is_deal=model.predict(X))

## 5. Сохранить CSV

Оставляем прогнозы модели для незаблокированных объявлений. Для заблокированных ставим `is_deal = 0`.

In [8]:
submission = predictions.copy()
submission.loc[submission["listing_id"].isin(blocked_ids), "is_deal"] = 0

assert submission.columns.tolist() == ["listing_id", "is_deal"]
assert submission["listing_id"].is_unique
assert set(submission["listing_id"]) == set(listings.index)
assert submission["is_deal"].isin([0, 1]).all()

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Сохранено {len(submission)} объявлений в {OUTPUT_PATH}")
submission.head()

Сохранено 15274 объявлений в submission_logreg_without_cv_features.csv
Out[8]: 
   listing_id  is_deal
0      132065        1
1      131440        0
2      117997        1
3      129810        1
4      111780        0
